In [ ]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
from synth_extract.agents.classification import (  # noqa: E402
    ClassificationFailure,
    ClassificationResult,
    ClassificationOutcome,
    PaperClassifier,
    FullTextClassifier
)
from synth_extract.agents.llm import LLMBackend  # noqa: E402


from pydantic import BaseModel, ConfigDict, Field
from synth_extract.agents.classification.schemas import CompletionMetadata

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import asyncio

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [ ]:
from typing import Literal

ClassificationCategory = Literal[
    "polymer_synthesis",
    "polymer_modification_combination",
    "polymer_composite_formulation",
]

class ClassificationResult(BaseModel):
    """A binary paper-classification result with an optional category."""

    model_config = ConfigDict(extra="forbid", strict=True)

    label: bool = Field(
        description="Whether the paper matches the classification criteria."
    )

    category: ClassificationCategory | None = Field(
        description=(
            "Material-creation category when label is true; "
            "null when label is false."
        )
    )

    metadata: CompletionMetadata

In [ ]:
import json
from pathlib import Path
from typing import Any

from pydantic import ValidationError


class FullTextCategoryClassifier(PaperClassifier):
    """Classify full text and return a label with an optional category."""

    def __init__(
        self,
        backend: LLMBackend,
        system_prompt_path: str | Path | None = None,
        user_template_path: str | Path | None = None,
    ) -> None:
        super().__init__(
            backend=backend,
            system_prompt_path=(
                system_prompt_path or _FULL_TEXT_SYSTEM_PROMPT_PATH
            ),
            user_template_path=(
                user_template_path or _FULL_TEXT_USER_TEMPLATE_PATH
            ),
        )

    @staticmethod
    def response_format() -> dict[str, Any]:
        """Return the label-and-category response schema."""
        return {
            "type": "json_schema",
            "json_schema": {
                "name": "paper_category_classification",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "boolean",
                            "description": (
                                "Whether the paper matches the "
                                "classification criteria."
                            ),
                        },
                        "category": {
                            "anyOf": [
                                {
                                    "type": "string",
                                    "enum": [
                                        "polymer_synthesis",
                                        "polymer_modification_combination",
                                        "polymer_composite_formulation",
                                    ],
                                },
                                {
                                    "type": "null",
                                },
                            ],
                            "description": (
                                "Material-creation category when label is true; "
                                "null when label is false."
                            ),
                        },
                    },
                    "required": [
                        "label",
                        "category",
                    ],
                    "additionalProperties": False,
                },
            },
        }

    def build_messages(
        self,
        fulltext: str,
    ) -> list[dict[str, str]]:
        """Build the system and user messages."""
        fulltext = fulltext.strip()

        return [
            {
                "role": "system",
                "content": self._system_prompt,
            },
            {
                "role": "user",
                "content": self._user_template.format(
                    fulltext=fulltext,
                ),
            },
        ]

    def render_prompt(self, fulltext: str) -> str:
        """Return a readable rendering of the messages."""
        return self._render_messages(
            self.build_messages(fulltext)
        )

    @staticmethod
    def _validate_input(
        fulltext: str,
    ) -> ClassificationFailure | None:
        """Return an input failure when the full text is empty."""
        if not fulltext.strip():
            return ClassificationFailure(
                error_type="input",
                message="Full text must be non-empty.",
            )

        return None

    @classmethod
    def _parse_completion(
        cls,
        completion: Any,
    ) -> ClassificationOutcome:
        """Parse the label, category, and completion metadata."""
        if not completion.choices:
            return ClassificationFailure(
                error_type="empty_response",
                message=(
                    "The provider returned no completion choices."
                ),
            )

        choice = completion.choices[0]

        if choice.finish_reason == "length":
            return ClassificationFailure(
                error_type="truncated",
                message=(
                    "The classification response reached "
                    "the token limit."
                ),
            )

        message = choice.message
        refusal = getattr(message, "refusal", None)

        if refusal:
            return ClassificationFailure(
                error_type="refusal",
                message=refusal,
            )

        content = message.content

        if not content:
            return ClassificationFailure(
                error_type="empty_response",
                message=(
                    "The provider returned an empty response."
                ),
            )

        try:
            payload = json.loads(content)

            if not isinstance(payload, dict):
                raise ValueError(
                    "The response must be a JSON object."
                )

            return ClassificationResult.model_validate(
                {
                    **payload,
                    "metadata": cls._completion_metadata(
                        completion
                    ),
                }
            )

        except (
            json.JSONDecodeError,
            ValidationError,
            ValueError,
        ) as exc:
            return ClassificationFailure(
                error_type="invalid_response",
                message=(
                    "The response did not match "
                    f"ClassificationResult: {exc}"
                ),
            )

    def classify_raw(
        self,
        fulltext: str,
    ) -> Any | ClassificationFailure:
        """Return the raw synchronous completion."""
        input_failure = self._validate_input(fulltext)

        if input_failure is not None:
            return input_failure

        try:
            messages = self.build_messages(fulltext)
        except Exception as exc:
            return self._request_failure(exc)

        return self._classify_messages_raw(messages)

    def classify(
        self,
        fulltext: str,
    ) -> ClassificationOutcome:
        """Synchronously classify one paper."""
        completion = self.classify_raw(fulltext)

        if isinstance(completion, ClassificationFailure):
            return completion

        return self._parse_completion(completion)

    async def aclassify_raw(
        self,
        fulltext: str,
    ) -> Any | ClassificationFailure:
        """Return the raw asynchronous completion."""
        input_failure = self._validate_input(fulltext)

        if input_failure is not None:
            return input_failure

        try:
            messages = self.build_messages(fulltext)
        except Exception as exc:
            return self._request_failure(exc)

        return await self._aclassify_messages_raw(messages)

    async def aclassify(
        self,
        fulltext: str,
    ) -> ClassificationOutcome:
        """Asynchronously classify one paper."""
        completion = await self.aclassify_raw(fulltext)

        if isinstance(completion, ClassificationFailure):
            return completion

        return self._parse_completion(completion)

In [ ]:
host="127.0.0.1"
port="8000"
base_url=f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens=8192
extra_body = {"chat_template_kwargs":{"enable_thinking":True}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=300,
    max_tokens=max_tokens,
    extra_body=extra_body,
)

In [ ]:
base_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/del_classification")
system_prompt_path = base_path / "s1.md"
user_prompt_path = base_path / "user_prompt.md"

In [ ]:
classifier = FullTextCategoryClassifier(backend=backend,
system_prompt_path=system_prompt_path,
user_template_path=user_prompt_path
)

# classifier = FullTextClassifier(backend=backend,
# system_prompt_path=system_prompt_path,
# user_template_path=user_prompt_path
# )

In [ ]:
classifier.health_check()

In [ ]:
dev_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/development_set")
label_path = dev_path / "dataset_labels_category.csv"

In [ ]:
df = pd.read_csv(label_path)

In [ ]:
df.head()

In [ ]:
def dataset_stats(data, name):
    total = len(data)
    positives = (data["label"] == 1).sum()
    negatives = (data["label"] == 0).sum()

    uncertain = data["uncertain"].notna().sum()
    certain = data["uncertain"].isna().sum()

    certain_data = data[data["uncertain"].isna()]
    certain_positives = (certain_data["label"] == 1).sum()
    certain_negatives = (certain_data["label"] == 0).sum()

    return {
        "Dataset": name,
        "N": total,
        
        "Positive": positives,
        "Negative": negatives,
        "Uncertain": uncertain,
        "Certain": certain,
        "N after removing uncertain": len(certain_data),
        "Positive after removing uncertain": certain_positives,
        "Negative after removing uncertain": certain_negatives,
    }


stats = [
    dataset_stats(df, "Full"),
    dataset_stats(df[df["split"] == "train"], "Train"),
    dataset_stats(df[df["split"] == "test"], "Test"),
]

dataset_stats_table = pd.DataFrame(stats).set_index("Dataset")

dataset_stats_table

In [ ]:
# df = df[df["canonical_source"].isin(["wiley", "springer_nature"])]
# print(f"Number of rows after filtering: {len(df)}")

In [ ]:
df

In [ ]:
system_prompt_path = base_path / "s4.4.category.md"
classifier.update_prompt_paths(system_prompt_path=system_prompt_path)

In [ ]:
print(classifier.system_prompt())

In [ ]:
classifier.backend.extra_body = {"chat_template_kwargs":{"enable_thinking":True}}

In [ ]:
classifier.backend.config()

In [83]:
# Pick a paper
uid = "ID000034858"

# Find its row
row = df.loc[df["paper_uid"] == uid].iloc[0]

source = row["canonical_source"]

# Construct full-text path
fulltext_path = (
    Path(dev_path)
    / source
    / uid
    / f"{uid}.md"
)

print("Source:", source)
print("Full text:", fulltext_path)

if not fulltext_path.exists():
    raise FileNotFoundError(fulltext_path)

# Load Markdown
full_text = fulltext_path.read_text(encoding="utf-8")

print(f"Characters: {len(full_text):,}")

# Classify
result = classifier.classify(full_text)

result

Source: wiley
Full text: /nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/development_set/wiley/ID000034858/ID000034858.md
Characters: 23,765


ClassificationResult(label=True, category='polymer_synthesis', metadata=CompletionMetadata(model='qwen3.6-27b', created=1788280841, finish_reason='stop', stop_reason=None, reasoning='The user wants me to classify a scientific paper about a polymer material.\nI need to determine the binary `label` (true/false) and the `category` (if true).\n\nFirst, check Hard Exclusions:\n- purely computational? No.\n- review/perspective? No.\n- depolymerization/recycling? No.\n- sequence-defined biological macromolecules? No.\n- process-only studies? No.\n- engineered into macroscopic article/device? The paper discusses shape memory polymers (SMPs) and tests them in bulk form (films/samples) for shape memory effect, rheology, XRD, DMA. It\'s not a device or patterned structure. It\'s a bulk polymer material study.\n\nCheck Material Creation:\n- "P1 with a molecular weight (MW) of ≈ 3.0 × 10^5 g mol−1 was synthesized by ring-opening metathesis polymerization."\n- They synthesize a new polymer: hemiphas

In [84]:
print(result.metadata.reasoning)

The user wants me to classify a scientific paper about a polymer material.
I need to determine the binary `label` (true/false) and the `category` (if true).

First, check Hard Exclusions:
- purely computational? No.
- review/perspective? No.
- depolymerization/recycling? No.
- sequence-defined biological macromolecules? No.
- process-only studies? No.
- engineered into macroscopic article/device? The paper discusses shape memory polymers (SMPs) and tests them in bulk form (films/samples) for shape memory effect, rheology, XRD, DMA. It's not a device or patterned structure. It's a bulk polymer material study.

Check Material Creation:
- "P1 with a molecular weight (MW) of ≈ 3.0 × 10^5 g mol−1 was synthesized by ring-opening metathesis polymerization."
- They synthesize a new polymer: hemiphasmid side-chain polynorbornene (P1).
- This qualifies as material creation.

Check Characterization/Property Measurement:
- They characterize P1 using 1D XRD, 2D XRD, polarized optical microscopy, rh

In [72]:
max_parallel_requests = 8
async_semaphore = asyncio.Semaphore(max_parallel_requests)


async def classify_row_async(
    n,
    row,
) -> ClassificationResult | ClassificationFailure | None:
    uid = row["paper_uid"]
    source = row["canonical_source"]
    true_label = int(row["label"])
    fulltext_path = Path(dev_path) / source / uid / f"{uid}.md"

    if not fulltext_path.exists():
        print(
            f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
            f"True: {true_label} | Prediction: None | "
            f"Tokens: None | Missing: {fulltext_path}"
        )
        return None

    try:
        full_text = fulltext_path.read_text(encoding="utf-8")

        async with async_semaphore:
            result = await classifier.aclassify(full_text)

        if isinstance(result, ClassificationResult):
            prediction = int(result.label)
            usage = result.metadata.usage

            token_summary = (
                f"input={usage.prompt_tokens}, "
                f"output={usage.completion_tokens}, "
                f"total={usage.total_tokens}"
                if usage is not None
                else "unavailable"
            )

            print(
                f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
                f"True: {true_label} | Prediction: {prediction} | "
                f"Tokens: {token_summary}"
            )
            return result

        if isinstance(result, ClassificationFailure):
            print(
                f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
                f"True: {true_label} | Prediction: None | "
                f"Tokens: None | Failure: "
                f"{result.error_type}: {result.message}"
            )
            return result

        raise TypeError(
            f"Unexpected classification result: {type(result).__name__}"
        )

    except Exception as exc:
        print(
            f"[{n}/{len(df)}] Source: {source} | UID: {uid} | "
            f"True: {true_label} | Prediction: None | "
            f"Tokens: None | Error: {type(exc).__name__}: {exc}"
        )
        return None


tasks = [
    asyncio.create_task(classify_row_async(n, row))
    for n, (_, row) in enumerate(df.iterrows(), start=1)
]

classification_outcomes: list[
    ClassificationResult | ClassificationFailure | None
] = await asyncio.gather(*tasks)

[2/120] Source: arxiv | UID: ID001096952 | True: 0 | Prediction: 0 | Tokens: input=42921, output=576, total=43497
[4/120] Source: arxiv | UID: ID001097924 | True: 0 | Prediction: 0 | Tokens: input=9817, output=830, total=10647
[5/120] Source: arxiv | UID: ID001098054 | True: 0 | Prediction: 0 | Tokens: input=12101, output=853, total=12954
[1/120] Source: arxiv | UID: ID001098281 | True: 0 | Prediction: 0 | Tokens: input=13949, output=1107, total=15056
[7/120] Source: arxiv | UID: ID001100563 | True: 0 | Prediction: 0 | Tokens: input=9869, output=1447, total=11316
[3/120] Source: arxiv | UID: ID001098560 | True: 0 | Prediction: 0 | Tokens: input=9014, output=1525, total=10539
[6/120] Source: arxiv | UID: ID000852121 | True: 0 | Prediction: 0 | Tokens: input=8876, output=1603, total=10479
[9/120] Source: arxiv | UID: ID001098081 | True: 0 | Prediction: 0 | Tokens: input=15294, output=1063, total=16357
[8/120] Source: arxiv | UID: ID001098961 | True: 1 | Prediction: 1 | Tokens: input=1536

In [73]:
# Evaluate only successful classifications
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

y_true = df.loc[valid, "label"].astype(int)
y_pred = [
    int(outcome.label)
    for outcome in classification_outcomes
    if isinstance(outcome, ClassificationResult)
]

classification_failures = sum(
    isinstance(outcome, ClassificationFailure)
    for outcome in classification_outcomes
)
missing_or_errors = sum(
    outcome is None
    for outcome in classification_outcomes
)
print(f"Actual negatives:        {(y_true == 0).sum()}")
print(f"Actual positives:        {(y_true == 1).sum()}")

print(f"Evaluated:               {len(y_pred)}/{len(classification_outcomes)}")
print(f"Classification failures: {classification_failures}")
print(f"Missing/errors:          {missing_or_errors}")
print()

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

Actual negatives:        64
Actual positives:        56
Evaluated:               120/120
Classification failures: 0
Missing/errors:          0

Accuracy:  0.858
Precision: 0.898
Recall:    0.786
F1:        0.838

Confusion matrix:
[[59  5]
 [12 44]]


In [74]:
# Evaluate only successful classifications for non-uncertain papers
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

non_uncertain = df["uncertain"].isna().to_numpy()

eval_mask = [
    is_valid and is_non_uncertain
    for is_valid, is_non_uncertain in zip(valid, non_uncertain)
]

y_true = df.loc[eval_mask, "label"].astype(int)

y_pred = [
    int(outcome.label)
    for outcome, keep in zip(classification_outcomes, eval_mask)
    if keep
]

classification_failures = sum(
    isinstance(outcome, ClassificationFailure) and is_non_uncertain
    for outcome, is_non_uncertain in zip(classification_outcomes, non_uncertain)
)

missing_or_errors = sum(
    outcome is None and is_non_uncertain
    for outcome, is_non_uncertain in zip(classification_outcomes, non_uncertain)
)

print("Non-uncertain papers only")
print(f"Actual negatives:        {(y_true == 0).sum()}")
print(f"Actual positives:        {(y_true == 1).sum()}")

print(f"Evaluated:               {len(y_pred)}/{non_uncertain.sum()}")
print(f"Classification failures: {classification_failures}")
print(f"Missing/errors:          {missing_or_errors}")
print()

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

Non-uncertain papers only
Actual negatives:        49
Actual positives:        39
Evaluated:               88/88
Classification failures: 0
Missing/errors:          0

Accuracy:  0.920
Precision: 0.944
Recall:    0.872
F1:        0.907

Confusion matrix:
[[47  2]
 [ 5 34]]


In [75]:
for split_name in ["train", "test"]:
    split_mask = df["split"] == split_name

    valid = [
        split_mask.iloc[i] and isinstance(outcome, ClassificationResult)
        for i, outcome in enumerate(classification_outcomes)
    ]

    y_true = df.loc[valid, "label"].astype(int)

    y_pred = [
        int(outcome.label)
        for i, outcome in enumerate(classification_outcomes)
        if split_mask.iloc[i] and isinstance(outcome, ClassificationResult)
    ]

    classification_failures = sum(
        split_mask.iloc[i] and isinstance(outcome, ClassificationFailure)
        for i, outcome in enumerate(classification_outcomes)
    )

    missing_or_errors = sum(
        split_mask.iloc[i] and outcome is None
        for i, outcome in enumerate(classification_outcomes)
    )

    total = split_mask.sum()

    print("=" * 60)
    print(split_name.upper())
    print("=" * 60)

    print(f"Actual negatives:        {(y_true == 0).sum()}")
    print(f"Actual positives:        {(y_true == 1).sum()}")

    print(f"Evaluated:               {len(y_pred)}/{total}")
    print(f"Classification failures: {classification_failures}")
    print(f"Missing/errors:           {missing_or_errors}")
    print()

    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"F1:        {f1_score(y_true, y_pred):.3f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()

TRAIN
Actual negatives:        53
Actual positives:        47
Evaluated:               100/100
Classification failures: 0
Missing/errors:           0

Accuracy:  0.850
Precision: 0.881
Recall:    0.787
F1:        0.831

Confusion matrix:
[[48  5]
 [10 37]]

TEST
Actual negatives:        11
Actual positives:        9
Evaluated:               20/20
Classification failures: 0
Missing/errors:           0

Accuracy:  0.900
Precision: 1.000
Recall:    0.778
F1:        0.875

Confusion matrix:
[[11  0]
 [ 2  7]]



In [76]:
results = {}

for scope_name, scope_mask in {
    "Full": pd.Series(True, index=df.index),
    "Certain only": df["uncertain"].isna(),
}.items():

    for split_name in ["train", "test"]:
        split_mask = df["split"] == split_name
        mask = scope_mask & split_mask

        valid = [
            mask.iloc[i] and isinstance(outcome, ClassificationResult)
            for i, outcome in enumerate(classification_outcomes)
        ]

        y_true = df.loc[valid, "label"].astype(int)

        y_pred = [
            int(outcome.label)
            for i, outcome in enumerate(classification_outcomes)
            if mask.iloc[i] and isinstance(outcome, ClassificationResult)
        ]

        results[(scope_name, split_name.capitalize())] = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        }

metrics_table = pd.DataFrame(results)

# Optional: round for cleaner display
metrics_table = metrics_table.round(3)

metrics_table

Full        Certain only     
           Train   Test        Train Test
Accuracy   0.850  0.900        0.903  1.0
Precision  0.881  1.000        0.935  1.0
Recall     0.787  0.778        0.853  1.0
F1         0.831  0.875        0.892  1.0

In [77]:
# Evaluate only successful classifications
valid = [
    isinstance(outcome, ClassificationResult)
    for outcome in classification_outcomes
]

y_true = df.loc[valid, "label"].astype(int)

y_pred = [
    int(outcome.label)
    for outcome in classification_outcomes
    if isinstance(outcome, ClassificationResult)
]

categories = [
    outcome.category
    for outcome in classification_outcomes
    if isinstance(outcome, ClassificationResult)
]

# Check label/category consistency
category_errors = []

for i, outcome in enumerate(classification_outcomes):
    if not isinstance(outcome, ClassificationResult):
        continue

    if outcome.label and outcome.category is None:
        category_errors.append(
            (i, "label=True but category=None")
        )

    if not outcome.label and outcome.category is not None:
        category_errors.append(
            (i, f"label=False but category={outcome.category!r}")
        )

print(f"Category consistency errors: {len(category_errors)}")

for error in category_errors[:20]:
    print(error)

# Category counts
category_counts = pd.Series(categories).value_counts(dropna=False)

print("\nCategory counts:")
print(category_counts)

Category consistency errors: 0

Category counts:
NaN                                 71
polymer_composite_formulation       28
polymer_synthesis                   14
polymer_modification_combination     7
Name: count, dtype: int64


In [78]:
# ### Update the table

# column_name = "qwen_s4.4_res_K3"
# df[column_name] = pd.array(
#     [
#         int(outcome.label)
#         if isinstance(outcome, ClassificationResult)
#         else pd.NA
#         for outcome in classification_outcomes
#     ],
#     dtype="Int64",
# )

# # label_path = dev_path / "dataset_labels_del.csv"
# df.to_csv(label_path, index=False)

In [ ]:
# ### Update the table

# label_column = "qwen_s4.4_category_res_K3"
# category_column = "qwen_s4.4_category_res_catcol_K3"

# df[label_column] = pd.array(
#     [
#         int(outcome.label)
#         if isinstance(outcome, ClassificationResult)
#         else pd.NA
#         for outcome in classification_outcomes
#     ],
#     dtype="Int64",
# )

# df[category_column] = pd.array(
#     [
#         outcome.category
#         if isinstance(outcome, ClassificationResult)
#         else pd.NA
#         for outcome in classification_outcomes
#     ],
#     dtype="string",
# )

# df.to_csv(label_path, index=False)

In [80]:
df.head()

,paper_id,paper_uid,title,canonical_source,journal,identifier_link,qwen,gemma,split,label,...,qwen_s4.4_category_K2,qwen_s4.4_category_catcol_K2,qwen_s4.4_category_K3,qwen_s4.4_category_catcol_K3,qwen_s4.4_category_res_K1,qwen_s4.4_category_res_catcol_K1,qwen_s4.4_category_res_K2,qwen_s4.4_category_res_catcol_K2,qwen_s4.4_category_res_K3,qwen_s4.4_category_res_catcol_K3
0,1098281,ID001098281,Local Mechanical Description of an Elastic Fold,arxiv,arxiv,https://arxiv.org/abs/1808.04892,0,0,train,0,...,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>
1,1096952,ID001096952,Multiscale Modeling and Coarse Graining of Pol...,arxiv,arxiv,https://arxiv.org/abs/0911.1001,0,0,train,0,...,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>
2,1098560,ID001098560,Quantum yield enhancement in BDMO-PPV,arxiv,arxiv,https://arxiv.org/abs/1910.11468,0,0,train,0,...,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>
3,1097924,ID001097924,Ordered qausi-two-dimensional structure of nan...,arxiv,arxiv,https://arxiv.org/abs/1701.05659,0,0,train,0,...,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>
4,1098054,ID001098054,Fake $μ$s: A cautionary tail of shear-thinning...,arxiv,arxiv,https://arxiv.org/abs/1708.09222,0,0,train,0,...,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>
